<div style="background: linear-gradient(135deg, #1a1a2e 0%, #16213e 50%, #0f3460 100%); padding: 40px 30px; border-radius: 12px; margin-bottom: 10px;">
  <h1 style="color: #e0e0e0; font-size: 2.2em; margin: 0 0 10px 0; font-weight: 700; letter-spacing: 1px;">2D Gradient Echo (GRE) Sequence</h1>
  <h2 style="color: #a0c4ff; font-size: 1.2em; margin: 0 0 20px 0; font-weight: 400;">Simulation & Reconstruction with MRzero and PyPulseq</h2>
  <hr style="border: 1px solid #a0c4ff; opacity: 0.3; margin: 15px 0;"/>
  <p style="color: #c0c0c0; margin: 0; font-size: 0.95em;">Advanced MRI | Prof. Dr.-Ing. Markus Zimmermann</p>
</div>

## 📋 Overview

This notebook walks you through the **full pipeline** of an MRI sequence simulation — from sequence design to image reconstruction — using two powerful Python frameworks:

| Tool | Role |
|------|------|
| [**PyPulseq**](https://pypulseq.readthedocs.io) | Sequence design: define RF pulses, gradients, and ADC events in a vendor-agnostic format |
| [**MRzero**](https://mrzero-core.readthedocs.io) | Physics simulation: propagate magnetization through the sequence using Bloch equations |

By the end of this notebook, you will have:
1. Built a 2D GRE sequence from scratch
2. Simulated signal acquisition through a built-in brain phantom
3. Reconstructed a 2D image from the simulated k-space data


## 🎯 Learning Objectives

After completing this notebook, you should have a good understanding of:

- ✅ Cartesian sampling
- ✅ How a sequence is programmed
- ✅ The role of each gradient axis (slice, phase, readout) in a 2D GRE sequence
- ✅ Applying a 2D inverse FFT to reconstruct an image from k-space

---
## 🧠 Background: The Gradient Echo Sequence

### What is a Gradient Echo?

A **gradient echo** is formed when dephasing caused by a readout pre-phasing gradient is exactly reversed by the readout gradient. Unlike a spin echo, **no 180° refocusing pulse** is used — the echo is purely gradient-driven. This makes GRE sequences very fast, but also sensitive to field inhomogeneities (T2\* decay rather than T2).

### Sequence Timing 

90° Pulse --> TE --> ADC

### K-space and Cartesian Sampling

Each acquisition line fills **one row** of k-space. The phase-encoding gradient (Gy) steps through k-space line by line:

- The **pre-phaser gradient** moves to the start of each line (edge of k-space in kx)
- The **readout gradient** sweeps across the line 
- The **phase rewind gradient** returns ky to centre before spoiling

In [7]:
import matplotlib.pyplot as plt
import numpy as np
from ipywidgets import IntSlider, interact

N = 128
from skimage.data import shepp_logan_phantom
from skimage.transform import resize

phantom = resize(shepp_logan_phantom(), (N, N))
full_kspace = np.fft.fftshift(np.fft.fft2(np.fft.ifftshift(phantom)))


def show_kspace(n_lines):
    partial = np.zeros_like(full_kspace)
    centre = N // 2
    indices = sorted(range(N), key=lambda i: abs(i - centre))
    for i in indices[:n_lines]:
        partial[i, :] = full_kspace[i, :]

    image = np.abs(np.fft.fftshift(np.fft.ifft2(np.fft.ifftshift(partial))))

    fig, axes = plt.subplots(1, 2, figsize=(9, 4))
    axes[0].imshow(np.log1p(np.abs(partial)), cmap="gray")
    axes[0].set_title(f"K-space ({n_lines}/{N} lines)")
    axes[0].axis("off")
    axes[1].imshow(image, cmap="gray")
    axes[1].set_title("Reconstructed image")
    axes[1].axis("off")
    plt.tight_layout()
    plt.show()


interact(show_kspace, n_lines=IntSlider(min=1, max=N, step=1, value=1))

interactive(children=(IntSlider(value=1, description='n_lines', max=128, min=1), Output()), _dom_classes=('wid…

<function __main__.show_kspace(n_lines)>

---
## 📦 Part 1: Import Libraries

We import the following packages:

- **`numpy`** — numerical arrays and FFT operations
- **`torch`** — required by MRzero for tensor operations
- **`matplotlib`** — plotting and image display
- **`pypulseq`** — sequence design and timing engine
- **`MRzeroCore`** — Bloch simulation engine

In [ ]:
import matplotlib.pyplot as plt
import MRzeroCore as mr0
import numpy as np
import pypulseq as pp
import torch

---
## 🔧 Part 2: Sequence Design

The `main()` function below builds a complete 2D GRE sequence using PyPulseq. The sequence is fully parametric — you can adjust FOV, matrix size, flip angle, TE, and TR without rewriting the logic.

### Hardware Constraints (System Limits)

Before designing any sequence, we define the **scanner hardware limits**. These reflect typical clinical 1.5T or 3T scanner capabilities:

| Limit | Value | Meaning |
|-------|-------|---------|
| Max gradient | 28 mT/m | Peak gradient amplitude |
| Max slew rate | 150 T/m/s | Gradient switching speed (PNS limit) |
| RF ringdown | 20 µs | Wait after RF before gradients |
| RF dead time | 100 µs | Minimum delay before RF transmit |
| ADC dead time | 10 µs | Settling time before ADC opens |

### Sequence Construction Steps

The function performs the following steps in order:

1. **Sinc RF pulse** — slice-selective excitation with apodised sinc shape (Hanning window, TBW = 4)
2. **Readout gradient** — flat-top trapezoid; ADC samples during the flat top
3. **Pre-phaser + phase encode** — applied simultaneously to save time; shifts k-space to the correct starting position
4. **Slice rephaser** — refocuses dephasing caused by the slice-selection gradient
5. **TE/TR delay calculation** — precise timing to hit the desired echo time and repetition time
6. **Spoiler gradients** — crusher on Gx and Gz to destroy residual transverse magnetisation
7. **RF spoiling** — quadratic phase increment of 117° applied per TR
8. **Timing check** — PyPulseq validates all block timings before export

>  **Note on `phase_areas`:** The phase-encoding steps are symmetric around zero: from $-N_y/2 \cdot \Delta k_y$ to $+(N_y/2 - 1) \cdot \Delta k_y$. This ensures k-space is sampled symmetrically and the DC component (ky = 0) is captured near the middle of the acquisition.

### ▶️ Run the Sequence Builder

Call `main()` to construct the sequence and write it to `gre.seq`.

Setting `plot=True` will display the **sequence diagram** — a timing plot showing all RF, gradient, and ADC events for the first TR. This is a critical QA step: inspect it to verify:
- The RF pulse sits correctly within the slice-selection gradient
- The ADC window is centred on the readout gradient flat top
- TE and TR timings match your intended values

Setting `write_seq=True` saves the sequence to a `.seq` file — the **Pulseq open file format**, which can be transferred to real MRI scanners supporting Pulseq.

In [ ]:
def main(
    plot: bool = False,
    test_report: bool = False,
    write_seq: bool = False,
    fov: float | tuple[float, float] = 256e-3,
    n_x: int = 128,
    n_y: int = 128,
    flip_angle_deg: float = 10,
    slice_thickness: float = 3e-3,
    tr: float = 12e-3,
    te: float = 5e-3,
):
    fov_x, fov_y = (fov, fov) if isinstance(fov, (int, float)) else fov
    rf_spoiling_inc = 117

    # Set system limits
    system = pp.Opts(
        max_grad=28,
        grad_unit="mT/m",
        max_slew=150,
        slew_unit="T/m/s",
        rf_ringdown_time=20e-6,
        rf_dead_time=100e-6,
        adc_dead_time=10e-6,
    )

    # Create slice selection pulse and gradient

    rf, gz, _ = pp.make_sinc_pulse(
        flip_angle=np.deg2rad(flip_angle_deg),
        duration=3e-3,
        slice_thickness=slice_thickness,
        apodization=0.42,
        time_bw_product=4,
        system=system,
        return_gz=True,
        delay=system.rf_dead_time,
        use="excitation",
    )

    # Define other gradients and ADC events
    delta_kx = 1 / fov_x
    delta_ky = 1 / fov_y
    gx = pp.make_trapezoid(
        channel="x",
        flat_area=n_x * delta_kx,
        flat_time=3.2e-3,
        system=system,
    )
    adc = pp.make_adc(
        num_samples=n_x,
        duration=gx.flat_time,
        delay=gx.rise_time,
        system=system,
    )
    gx_pre = pp.make_trapezoid(
        channel="x",
        area=-gx.area / 2,
        duration=1e-3,
        system=system,
    )
    gz_reph = pp.make_trapezoid(
        channel="z",
        area=-gz.area / 2,
        duration=1e-3,
        system=system,
    )
    phase_areas = (np.arange(n_y) - n_y / 2) * delta_ky

    # Gradient spoiling
    gx_spoil = pp.make_trapezoid(channel="x", area=2 * n_x * delta_kx, system=system)
    gz_spoil = pp.make_trapezoid(channel="z", area=4 / slice_thickness, system=system)

    EPS = 1e-12

    # Calculate timing
    te_delay = (
        te
        - (pp.calc_duration(gz, rf) - pp.calc_rf_center(rf)[0] - rf.delay)
        - pp.calc_duration(gx_pre)
        - pp.calc_duration(gx) / 2
        - EPS
    )
    te_delay = np.ceil(te_delay / seq.grad_raster_time) * seq.grad_raster_time

    tr_delay = (
        tr
        - pp.calc_duration(gz, rf)
        - pp.calc_duration(gx_pre)
        - pp.calc_duration(gx)
        - te_delay
    )
    tr_delay = np.ceil(tr_delay / seq.grad_raster_time) * seq.grad_raster_time

    assert np.all(te_delay >= 0)
    assert np.all(tr_delay >= pp.calc_duration(gx_spoil, gz_spoil))

    rf_phase = 0
    rf_inc = 0

    # Loop over phase encodes and define sequence blocks
    for i_phase in range(n_y):
        rf.phase_offset = rf_phase / 180 * np.pi
        adc.phase_offset = rf_phase / 180 * np.pi
        rf_inc = divmod(rf_inc + rf_spoiling_inc, 360.0)[1]
        rf_phase = divmod(rf_phase + rf_inc, 360.0)[1]

        seq.add_block(rf, gz)
        gy_pre = pp.make_trapezoid(
            channel="y",
            area=phase_areas[i_phase],
            duration=pp.calc_duration(gx_pre),
            system=system,
        )
        seq.add_block(gx_pre, gy_pre, gz_reph)
        seq.add_block(pp.make_delay(te_delay))
        seq.add_block(gx, adc)
        gy_pre.amplitude = -gy_pre.amplitude
        seq.add_block(pp.make_delay(tr_delay), gx_spoil, gy_pre, gz_spoil)

    ok, error_report = seq.check_timing()
    if ok:
        print("Timing check passed successfully")
    else:
        print("Timing check failed. Error listing follows:")
        [print(e) for e in error_report]

    if test_report:
        print(seq.test_report())

    if plot:
        seq.plot(time_range=(0.0, tr))

    seq.set_definition("FOV", [fov_x, fov_y, slice_thickness])
    seq.set_definition("Name", "gre")

    if write_seq:
        seq.write("2D GRE Cartesian.seq")

    return seq


if __name__ == "__main__":
    main(plot=True, write_seq=True)

---
## 🖥️ Part 3: Simulation & Reconstruction

### Bloch Simulation with MRzero

MRzero simulates the MR signal by propagating the magnetisation vector $\mathbf{M} = (M_x, M_y, M_z)$ through the sequence using the **Bloch equations**:

$$\frac{d\mathbf{M}}{dt} = \gamma \mathbf{M} \times \mathbf{B} - \frac{M_x \hat{x} + M_y \hat{y}}{T_2} - \frac{(M_z - M_0)\hat{z}}{T_1}$$

Each spatial location in the phantom contributes signal based on its **proton density (PD)**, **T1**, and **T2** values. The simulator:
1. Loads the `.seq` file to obtain the gradient waveforms and RF events
2. Applies them to a digital phantom (default: a simple 2D geometric phantom included with MRzero)
3. Returns the **complex signal** sampled at each ADC point

### K-space Reshaping

The raw signal is a 1D array of `n_x × n_y = 128 × 128 = 16,384` complex samples (one per ADC sample point). We reshape this into a 2D k-space matrix where:
- **Rows** correspond to phase-encoding steps (ky lines)
- **Columns** correspond to readout samples (kx)

### Image Reconstruction

For Cartesian k-space, reconstruction is a simple **2D inverse Fourier transform**:

$$\rho(x, y) = \mathcal{F}^{-1}\{S(k_x, k_y)\}$$

The final image is displayed as `|ρ(x,y)|` — the **magnitude image**, which is always positive and removes phase wraps from field inhomogeneity.

In [ ]:
# Load sequence
mr_seq = mr0.Sequence.import_file("2D GRE Cartesian.seq")
torch.set_default_device("cpu")
signal, _ = mr0.util.simulate("2D GRE Cartesian.seq", None)

signal = signal.numpy()
kspace = signal.reshape(128, 128)

image = np.fft.fftshift(np.fft.ifft2(np.fft.ifftshift(kspace)))

plt.figure()
plt.imshow(np.abs(image), cmap="gray")
plt.title("Reconstructed GRE Image")
plt.show()

---
## 📚 Summary

In this notebook you have:

1. **Designed** a 2D spoiled GRE sequence using PyPulseq, specifying RF pulses, gradients, and ADC events with hardware-constrained timing
2. **Exported** the sequence to the open Pulseq `.seq` format
3. **Simulated** MR signal acquisition through a digital phantom using MRzero's Bloch equation engine
4. **Reconstructed** a 2D image via the inverse 2D FFT